# 01. Knowledge-base pipeline (documented walkthrough)

This notebook **is the documented orchestration** of the ICD-11 knowledge-base build
(the same flow as `src/builders/run_kb_pipeline.py`).

## Pipeline overview

```
ICD-11 PDF  --chunker-->  icd11_chunks.json  --ingestion-->  ChromaDB vectors
```

| Step | Module | Input | Output |
|------|--------|-------|--------|
| 1. Chunking | `src/components/chunker.py` | `knowledge_base/icd_11/icd_11.pdf` | `knowledge_base/icd_11/icd11_chunks.json` |
| 2. Ingestion | `src/components/ingestion.py` | chunks JSON | `knowledge_base/icd_11/chroma_db/` |

Heavy lifting stays in those modules (PDF parsing / embeddings). This notebook
holds the **control flow, explanation, and inspection** that used to live only
as a thin CLI in `main.py`.

CLI still works for automation: `python src/builders/run_kb_pipeline.py`


In [4]:
# Path setup — works from repo root or notebooks/
import sys
from pathlib import Path

_cwd = Path.cwd().resolve()
root = None
for candidate in (_cwd, *_cwd.parents):
    if (candidate / "src" / "components" / "config.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError(
        "Could not locate project root. Open this notebook from the repo or notebooks/ folder."
    )

sys.path.insert(0, str(root / "src"))
sys.path.insert(0, str(root / "src" / "retriever"))

from components.config import (
    PROJECT_ROOT,
    PDF_PATH,
    CHUNKS_PATH,
    CHROMA_PATH,
    RAG_EVAL_SUBSET_PATH,
    RAG_DEV_SLICE_PATH,
    RAG_EVAL_LABELS,
    RETRIEVAL_SECTIONS,
    MOOD_DISORDER_PREFIXES,
    DATASET_PATH,
)

print("Project root:", PROJECT_ROOT)
print("PDF:         ", PDF_PATH, "| exists=", PDF_PATH.exists())
print("Chunks:      ", CHUNKS_PATH, "| exists=", CHUNKS_PATH.exists())
print("ChromaDB:    ", CHROMA_PATH, "| exists=", CHROMA_PATH.exists())
print("Final eval:  ", RAG_EVAL_SUBSET_PATH, "| exists=", RAG_EVAL_SUBSET_PATH.exists())
print("Dev slice:   ", RAG_DEV_SLICE_PATH, "| exists=", RAG_DEV_SLICE_PATH.exists())
print("Labels:      ", list(RAG_EVAL_LABELS))
print("Sections:    ", RETRIEVAL_SECTIONS)


Project root: /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026
PDF:          /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/icd_11/icd_11.pdf | exists= True
Chunks:       /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/icd_11/icd11_chunks.json | exists= True
ChromaDB:     /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/icd_11/chroma_db | exists= True
Final eval:   /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/datasets/processed/multiclass_eval.csv | exists= True
Dev slice:    /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/datasets/processed/multiclass_dev.csv | exists= True
Labels:       ['suicidal', 'depression', 'normal']
Sections:     ['Essential Features', 'Boundary with Normality']


## Configuration

Paths and knobs come from `src/components/config.py` so every entrypoint agrees
on the same PDF page range, chunk path, and embedding model.


In [5]:
# Paths already imported in the path-setup cell.
# Extra knobs for chunking / ingestion:
from components.config import (
    COLLECTION_NAME,
    EMBEDDING_MODEL,
    BATCH_SIZE,
    CONTENT_START_PAGE,
    CONTENT_END_PAGE,
)

print("Ready to run KB pipeline against knowledge_base/icd_11/")
print("Collection: ", COLLECTION_NAME)
print("Embed model:", EMBEDDING_MODEL)
print("Batch size: ", BATCH_SIZE)
print("PDF pages:  ", CONTENT_START_PAGE, "->", CONTENT_END_PAGE)
assert PDF_PATH.exists() or True  # PDF optional if chunks already exist
assert CHUNKS_PATH.exists(), f"Missing chunks JSON at {CHUNKS_PATH} — set RUN_CHUNKING=True or provide the file"
print("Chunks OK:", CHUNKS_PATH)


Ready to run KB pipeline against knowledge_base/icd_11/
Collection:  icd11_clinical
Embed model: FremyCompany/BioLORD-2023
Batch size:  64
PDF pages:   109 -> 694
Chunks OK: /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/icd_11/icd11_chunks.json


## Step 1: Chunking (PDF → JSON)

### What this step does
1. Extract text from the ICD-11 CDDR PDF for the clinical page range.
2. Parse disorder codes / section headings into structured chunk dicts.
3. Split oversized sections with word overlap.
4. Write `knowledge_base/icd_11/icd11_chunks.json`.

### Why it matters for RAG
Retrieval quality depends on chunk boundaries. We keep section-aware clinical
units (e.g. Essential Features, Boundary with Normality) rather than naive
fixed-size windows.

Set `RUN_CHUNKING = True` only when the PDF is present and you intend to rebuild.


In [6]:
from components.chunker import (
    run_chunking,
    MAX_CHUNK_WORDS,
    CHUNK_WORD_OVERLAP,
)

# Inspect existing artifacts by default. Flip to True only to rebuild from PDF.
RUN_CHUNKING = True

print(f"MAX_CHUNK_WORDS={MAX_CHUNK_WORDS}, OVERLAP={CHUNK_WORD_OVERLAP}")
print(f"RUN_CHUNKING={RUN_CHUNKING}")

if RUN_CHUNKING:
    if not PDF_PATH.exists():
        raise FileNotFoundError(f"Missing PDF at {PDF_PATH}")
    print("\n=== Step 1/2: Chunking ===")
    chunks = run_chunking(
        pdf_path=str(PDF_PATH),
        chunks_path=str(CHUNKS_PATH),
        start_page=CONTENT_START_PAGE,
        end_page=CONTENT_END_PAGE,
        max_words=MAX_CHUNK_WORDS,
        overlap_words=CHUNK_WORD_OVERLAP,
    )
    print(f"Wrote {len(chunks)} chunks -> {CHUNKS_PATH}")
else:
    print("Skipping chunking (inspect mode). Using existing JSON if present.")


MAX_CHUNK_WORDS=220, OVERLAP=35
RUN_CHUNKING=True

=== Step 1/2: Chunking ===
Extracting pages 109–694 from PDF …
pdftotext     : /opt/homebrew/bin/pdftotext
PDF path      : /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/icd_11/icd_11.pdf
File exists   : True
  Extracted 1,919,356 characters

Parsing into chunks …
  Raw chunks : 1000
  After split: 1576 (max_words=220, overlap=35)
  Dropped 7 junk/appendix chunks
  Final chunks: 1569

Top disorders by chunk count:
    56  6A00.Z  Disorder of intellectual development, unspecified
    42  6A8Z  Mood disorder, unspecified
    29  6A02  Autism spectrum disorder
    29  6E21  Mental and behavioural disorders associated with pregna
    24  6C4G.3  Intoxication due to unknown or unspecified psychoactive
    24  6C4G.4  Withdrawal due to unknown or unspecified psychoactive s
    22  6D11.5  Borderline pattern
    17  6B20.Z  Obsessive-compulsive disorder, unspecified
    16  6A05.Z  Attention deficit hyp

## Inspect chunks

After chunking (or if JSON already exists), inspect structure and content.
Each chunk typically includes disorder metadata plus clinical text fields used
for BM25 (`prompt_text`) and/or embedding (`embed_text` / `text`).


In [7]:
import json
from collections import Counter

if not CHUNKS_PATH.exists():
    raise FileNotFoundError(
        f"{CHUNKS_PATH} not found. Set RUN_CHUNKING=True or obtain the JSON artifact."
    )

with open(CHUNKS_PATH, encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Total chunks: {len(chunks)}")
print("Fields:", sorted(chunks[0].keys()))
print("\nTop sections:")
for name, n in Counter(c.get("section", "<none>") for c in chunks).most_common(8):
    print(f"  {n:5d}  {name}")

print("\n--- Examples ---")
for c in chunks[:3]:
    body = (c.get("prompt_text") or c.get("text") or "").replace("\n", " ")
    print(f"\n[{c.get('disorder_code')}] {c.get('disorder_name')} — {c.get('section')}")
    print(body[:320], "...")


Total chunks: 1569
Fields: ['disorder_code', 'disorder_name', 'domain', 'embed_text', 'section', 'source', 'text', 'word_count']

Top sections:
    421  Differential Diagnosis
    355  Overview
    274  Essential Features
    136  Additional Clinical Features
     98  Developmental Presentations
     87  Boundary with Normality
     83  Course Features
     64  Culture-Related Features

--- Examples ---

[6A01] Developmental speech and language disorders — Overview
Developmental speech and language disorders 6A01.0 Developmental speech sound disorder 6A01.1 Developmental speech fluency disorder 6A01.2 Developmental language disorder 6A01.Y Other specified developmental speech or language disorder 6A01.Z Developmental speech or language disorder, unspecified ...

[6A0Z] Neurodevelopmental disorder, unspecified. — Overview
Neurodevelopmental disorder, unspecified. Neurodevelopmental disorders 92 Clinical Descriptions and Diagnostic Requirements for ICD-11 Mental, Behavioural or Neurodeve

## Step 2: Ingestion (JSON → ChromaDB)

### What this step does
1. Load chunk JSON.
2. Embed each chunk with BioLORD-2023 (`sentence-transformers`).
3. Upsert vectors into a persistent Chroma collection.

### Why BioLORD
Domain embeddings improve dense retrieval over general-purpose models for
clinical wording in ICD-11 text.

This step can take several minutes on CPU. Use `REBUILD_CHROMA=True` only when
you need a clean re-index.


In [8]:
from components.ingestion import run_ingestion

# Inspect existing Chroma DB by default. Flip to True to re-embed.
RUN_INGESTION = True
REBUILD_CHROMA = True

print(f"RUN_INGESTION={RUN_INGESTION}, REBUILD_CHROMA={REBUILD_CHROMA}")

if RUN_INGESTION:
    if not CHUNKS_PATH.exists():
        raise FileNotFoundError(f"Missing chunks at {CHUNKS_PATH}")
    print("\n=== Step 2/2: Ingestion ===")
    run_ingestion(
        chunks_path=str(CHUNKS_PATH),
        chroma_path=str(CHROMA_PATH),
        collection_name=COLLECTION_NAME,
        embedding_model_name=EMBEDDING_MODEL,
        batch_size=BATCH_SIZE,
        rebuild=REBUILD_CHROMA,
    )
    print("Ingestion finished.")
else:
    print("Skipping ingestion (inspect mode).")
    print("Chroma path exists=", CHROMA_PATH.exists())


RUN_INGESTION=True, REBUILD_CHROMA=True

=== Step 2/2: Ingestion ===


/Users/omarhalasa/.pyenv/versions/3.12.0/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


No GPU detected — using CPU

Loading: FremyCompany/BioLORD-2023 …
Loaded on  : cpu
Output dims: 768

ChromaDB path: /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/icd_11/chroma_db
Collection 'icd11_clinical' already exists with 1569 vectors.
Deleted existing collection (rebuild=True).
Collection ready: 'icd11_clinical'  (0 vectors)
Loaded 1569 chunks from /Users/omarhalasa/Desktop/AI Group Project/Github/AI-group-project-2026/knowledge_base/icd_11/icd11_chunks.json

Ingesting 1569 chunks into 'icd11_clinical' …



Batches: 100%|██████████| 25/25 [01:01<00:00,  2.45s/it]


Done. Collection 'icd11_clinical' now has 1569 vectors.
Ingestion finished.


## Full orchestration (equivalent to `src/builders/run_kb_pipeline.py`)

The CLI `main()` is literally: optional chunking → ingestion → done.
The next cell mirrors that control flow so the thesis walkthrough stays in one place.


In [9]:
# Mirrors components.main:main() — flip flags above, then run this cell.
SKIP_CHUNKING = not RUN_CHUNKING
# (Chunking / ingestion already executed in the cells above when flags are True.)

print("Orchestration summary")
print(f"  skip_chunking = {SKIP_CHUNKING}")
print(f"  ran_ingestion = {RUN_INGESTION}")
print(f"  rebuild       = {REBUILD_CHROMA}")
print("\nCLI equivalents:")
print("  python src/builders/run_kb_pipeline.py")
print("  python src/builders/run_kb_pipeline.py --skip-chunking")
print("  python src/builders/run_kb_pipeline.py --rebuild")
print("\nNext: notebooks/02_multiclass_dataset_prep.ipynb")


Orchestration summary
  skip_chunking = False
  ran_ingestion = True
  rebuild       = True

CLI equivalents:
  python -m components.main
  python -m components.main --skip-chunking
  python -m components.main --rebuild

Next: notebooks/02_multiclass_dataset_prep.ipynb
